## Solar Thermal CalculationThe model uses a single temperature to characterize the solar field and relies on the EN12975 standard to characterize the solar thermal collector performance. The approach is based on Chapter 6 and Chapter 10 of “Solar Engineering of Thermal Processes”, by J.A Duffie and W.A. Beckman.  
The model can simulate the performance of the solar field collector, and then connect it to a thermal storage component or heat exchanger model to incorporate it in a model or a functional system.


First, some imports:

In [2]:
import pandas as pd

from pandapower.timeseries.data_sources.frame_data import DFData
from pandaprosumer.create import create_empty_prosumer_container, create_period
from pandaprosumer.create_controlled import (create_controlled_const_profile, create_controlled_solar_thermal)
from pandaprosumer.mapping import GenericMapping
from pandaprosumer.run_time_series import run_timeseries

In [3]:
data = pd.read_excel('data/senergy_nets_example_solar_thermal.xlsx')
data = data.iloc[0:96].copy()
data["time"] = pd.to_datetime(data["time"], format="%Y%m%d:%H%M")
data.set_index("time", inplace=True)
start = '2005-01-01 00:30:00'
end = '2005-01-05 00:29:59'
time_resolution_s = 3600        # 15 min
frequency = '60min'

Similarly to pandapower and pandapipes, the input for a time-series calculation has to be converted to the DFData format, which we imported before.

In [4]:
dur = pd.date_range(start=start, end=end, freq=frequency, tz='utc')
data.index = dur
data_input = DFData(data)
data.head()

,Beam Solar Radiation [W/m2],Diffuse Solar Radiation [W/m2],Ground Solar Radiation [W/m2],Radiation incidence angle [deg],Ambient temperature [C],Inlet temperature [C],Inlet mass flow rate [kg/h]
2005-01-01 00:30:00+00:00,0.0,0.0,0.0,0.0,-2.44,20.00,0
2005-01-01 01:30:00+00:00,0.0,0.0,0.0,0.0,-2.39,18.25,0
2005-01-01 02:30:00+00:00,0.0,0.0,0.0,0.0,-2.44,16.50,0
2005-01-01 03:30:00+00:00,0.0,0.0,0.0,0.0,-2.37,14.75,0
2005-01-01 04:30:00+00:00,0.0,0.0,0.0,0.0,-2.42,13.00,0


In [5]:
input_params = ['Beam Solar Radiation [W/m2]',
                 'Diffuse Solar Radiation [W/m2]',
                 'Ground Solar Radiation [W/m2]',
                 'Radiation incidence angle [deg]',
                 'Ambient temperature [C]',
                 'Inlet temperature [C]',
                 'Inlet mass flow rate [kg/h]']
result_params = [
                    "beam_solar_radiation_cp",
                    "diffuse_solar_radiation_cp",
                    "ground_solar_radiation_cp",
                    "radiation_incidence_angle_cp",
                    "ambient_temperature_cp",
                    "inlet_temperature_cp",
                    "inlet_mass_flow_rate_cp"
                ]

st_params = {}

prosumer = create_empty_prosumer_container()

period = create_period(prosumer, time_resolution_s, start, end, 'utc', 'default')

cp_index = create_controlled_const_profile(
    prosumer, input_params, result_params, data_input, period)

st_index = create_controlled_solar_thermal(prosumer, name="solar_thermal_plant", level=1, order=0, **st_params)


In [6]:
GenericMapping(
    prosumer,
    initiator_id=cp_index,
    initiator_column=["beam_solar_radiation_cp",
                    "diffuse_solar_radiation_cp",
                    "ground_solar_radiation_cp",
                    "radiation_incidence_angle_cp",
                    "ambient_temperature_cp",
                    "inlet_temperature_cp",
                    "inlet_mass_flow_rate_cp"],
    responder_id=st_index,
    responder_column=['beam_solar_radiation_w_m2',
                      'diffuse_solar_radiation_w_m2',
                      'ground_solar_radiation_w_m2',
                      'radiation_incidence_angle_deg',
                      'ambient_temperature_C',
                      'inlet_temperature_C',
                      'inlet_mass_flow_rate_kg_h',
                      ]
)


In [7]:
run_timeseries(prosumer)

100%|██████████| 96/96 [00:00<00:00, 411.57it/s]

1
1
[1, 1, 1]
gt: 0.0
0.19647473560517037
30.430007050528786
73.07948521739131
-2.4399999999999964
0.019647473560517038
2.9471210340775555
1
1
[1, 1, 1]
gt: 0.0
0.19647473560517037
30.410359576968272
71.55847605170388
-2.3900000000000032
0.019647473560517038
2.9471210340775555
1
1
[1, 1, 1]
gt: 0.0
0.19647473560517037
30.430007050528786
73.07948521739131
-2.4399999999999964
0.019647473560517038
2.9471210340775555
1
1
[1, 1, 1]
gt: 0.0
0.19647473560517037
30.402500587544065
70.95034745005876
-2.3700000000000023
0.019647473560517038
2.9471210340775555
1
1
[1, 1, 1]
gt: 0.0
0.19647473560517037
30.422148061104583
72.47096366627495
-2.4200000000000044
0.019647473560517038
2.9471210340775555
1
1
[1, 1, 1]
gt: 0.0
0.19647473560517037
30.319981198589893
64.57448686251469
-2.1600000000000024
0.019647473560517038
2.9471210340775555
1
1
[1, 1, 1]
gt: 0.0
0.19647473560517037
30.4457250293772
74.29699985898941
-2.480000000000007
0.019647473560517038
2.9471210340775555
1
1
[1, 1, 1]
gt: 303.64533522

In [8]:
res_df = prosumer.time_series.data_source.iloc[0].df
res_df.head(15)

,outlet_temperature_C,outlet_flow_rate_kg_h,energy_gain_W
2005-01-01 00:30:00+00:00,20.000000,0.0,-0.000000
2005-01-01 01:30:00+00:00,18.250000,0.0,-0.000000
2005-01-01 02:30:00+00:00,16.500000,0.0,-0.000000
2005-01-01 03:30:00+00:00,14.750000,0.0,-0.000000
2005-01-01 04:30:00+00:00,13.000000,0.0,-0.000000
2005-01-01 05:30:00+00:00,11.250000,0.0,-0.000000
2005-01-01 06:30:00+00:00,10.000000,0.0,-0.000000
2005-01-01 07:30:00+00:00,10.000000,0.0,0.000000
2005-01-01 08:30:00+00:00,19.921810,576.0,3730.974077
2005-01-01 09:30:00+00:00,25.937027,576.0,5016.419303
